# RAWG Dataset — Preprocessing
Minimal cleaning pipeline for the [atalaydenknalbant/rawg-games-dataset](https://huggingface.co/datasets/atalaydenknalbant/rawg-games-dataset) (890K rows, 54 cols).

**What this does:**
1. Drop useless columns (image URLs, color codes, screenshots, etc.)
2. Normalize nulls (`"nan"`, `""`, `NaN` → `NaN`)
3. **Filter to ~10-20K games with real data** (has rating, genre, release date, >10 ratings)
4. Parse nested JSON fields (`esrb_rating`, `added_by_status`)
5. Parse `ratings` distribution into separate columns
6. Fix dtypes (dates, numerics)
7. Compute derived features (completion rate, drop rate, rating delta)
8. Export clean parquet

In [1]:
import pandas as pd
import numpy as np
import json
import re
import ast

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)

## 1. Load & first look

In [3]:
# Adjust path to wherever you downloaded the CSV
df = pd.read_csv('rawg-games-dataset.csv', low_memory=False)
print(f"Shape: {df.shape}")
df.head(3)

Shape: (889793, 54)


,id,slug,name,released,background_image,rating,rating_top,ratings_count,reviews_text_count,added,playtime,suggestions_count,updated,reviews_count,saturated_color,dominant_color,platforms,stores,developers,genres,tags,publishers,esrb_rating,added_by_status,metacritic_url,ratings,clip,name_original,reddit_url,reactions,parents_count,background_image_additional,website,reddit_count,youtube_count,short_screenshots,creators_count,twitch_count,metacritic_platforms,screenshots_count,parent_platforms,description,description_raw,metacritic,achievements_count,alternative_names,parent_achievements_count,game_series_count,additions_count,movies_count,reddit_name,reddit_description,reddit_logo,tba
0,19103,half-life-2-lost-coast,Half-Life 2: Lost Coast,2005-10-27,https://media.rawg.io/media/games/b7b/b7b8381707152afc7d91f5d95de70e39.jpg,3.45,4.0,1164.0,3.0,11676.0,1.0,287.0,2025-07-19T10:25:47,1167.0,0f0f0f,0f0f0f,macOS|Linux|PC,Steam,Valve Software,Action,Singleplayer|Multiplayer|Atmospheric|Great Soundtrack|First-Person|Sci-fi|FP...,Valve,"{""id"":4,""name"":""Mature"",""slug"":""mature""}","{""beaten"":1066,""dropped"":178,""owned"":9637,""playing"":6,""toplay"":80,""yet"":709}",https://www.metacritic.com/game/pc/half-life-2-lost-coast,"{count: 550, id: 4, percent: 47.13, title: recommended}|{count: 263, id: 3, ...",NaN,Half-Life 2: Lost Coast,https://www.reddit.com/r/HalfLife/,"{""11"":1,""15"":1}",1.0,https://media.rawg.io/media/screenshots/259/259239f48f9e32210774b5641527071f...,http://www.half-life2.com,3988.0,1000000.0,"{id: -1, image: https://media.rawg.io/media/games/b7b/b7b8381707152afc7d91f5...",1.0,54.0,NaN,5.0,PC|Apple Macintosh|Linux,"Essentially a tech demo, “Half-Life 2: Lost Coast” sole purpose was to show ...","Essentially a tech demo, “Half-Life 2: Lost Coast” sole purpose was to show ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12020,left-4-dead-2,Left 4 Dead 2,2009-11-17,https://media.rawg.io/media/games/d58/d588947d4286e7b5e0e12e1bea7d9844.jpg,4.09,4.0,3355.0,25.0,17398.0,9.0,602.0,2025-07-08T21:52:17,3380.0,0f0f0f,0f0f0f,Xbox 360|Linux|PC|macOS,Steam|Xbox 360 Store,Valve Software|Turtle Rock Studios,Action|Shooter,Singleplayer|Steam Achievements|Multiplayer|Full controller support|Steam Cl...,Electronic Arts|Valve|Akella,"{""id"":4,""name"":""Mature"",""slug"":""mature""}","{""beaten"":2613,""dropped"":1236,""owned"":12885,""playing"":154,""toplay"":116,""yet""...",NaN,"{count: 1805, id: 4, percent: 53.4, title: recommended}|{count: 1070, id: 5,...",NaN,Left 4 Dead 2,NaN,"{""1"":3,""11"":4,""12"":1,""13"":1,""14"":1,""15"":1,""16"":2,""17"":1,""18"":1,""19"":1,""2"":3,...",NaN,https://media.rawg.io/media/screenshots/f01/f0147a2c654b7ace79a7fcebe94bf110...,http://www.l4d.com,NaN,1000000.0,"{id: -1, image: https://media.rawg.io/media/games/d58/d588947d4286e7b5e0e12e...",36.0,132.0,NaN,27.0,PC|Xbox|Apple Macintosh|Linux,Cooperative survival continues with a different set of characters. New survi...,Cooperative survival continues with a different set of characters. New survi...,89.0,166.0,L4D2,102.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,13536,portal,Portal,2007-10-09,https://media.rawg.io/media/games/7fa/7fa0b586293c5861ee32490e953a4996.jpg,4.49,5.0,4896.0,44.0,17582.0,4.0,303.0,2025-07-15T08:44:40,4940.0,0f0f0f,0f0f0f,macOS|PC|Android|PlayStation 3|Xbox 360|Linux|Nintendo Switch,Steam|Google Play,Valve Software|NVIDIA Lightspeed Studios,Action|Puzzle,Singleplayer|Steam Achievements|Atmospheric|Great Soundtrack|Story Rich|Firs...,Valve|Buka Entertainment|NVIDIA|CyberFront,"{""id"":3,""name"":""Teen"",""slug"":""teen""}","{""beaten"":5216,""dropped"":422,""owned"":11105,""playing"":86,""toplay"":285,""yet"":468}",NaN,"{count: 2960, id: 5, percent: 59.92, title: exceptional}|{count: 1671, id: 4...",NaN,Portal,https://www.reddit.com/r/Portal/,"{""1"":23,""10"":11,""11"":6,""12"":6,""2"":3,""3"":8,""4"":11,""5"":3,""6"":2,""7"":3,""8"":1}",NaN,https://media.rawg.io/media/screenshots/19d/19d3effb85e8f40d0b5b004fb5ab5c76...,http

In [4]:
# Quick overview of nulls and dtypes
info = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.notna().sum(),
    'null_pct': (df.isna().sum() / len(df) * 100).round(1),
    'sample': df.iloc[0].values
})
info

,dtype,non_null,null_pct,sample
id,int64,889793,0.0,19103
slug,str,889791,0.0,half-life-2-lost-coast
name,str,889753,0.0,Half-Life 2: Lost Coast
released,str,860336,3.3,2005-10-27
background_image,str,862399,3.1,https://media.rawg.io/media/games/b7b/b7b8381707152afc7d91f5d95de70e39.jpg
rating,float64,18109,98.0,3.45
rating_top,float64,20078,97.7,4.0
ratings_count,float64,57839,93.5,1164.0
reviews_text_count,float64,9471,98.9,3.0
added,float64,109395,87.7,11676.0


## 2. Drop useless columns

In [5]:
# Columns that are image URLs, color hex codes, or irrelevant for viz
DROP_COLS = [
    'background_image',
    'background_image_additional',
    'saturated_color',
    'dominant_color',
    'short_screenshots',
    'clip',                        # always null
    'description',                 # HTML version — keep description_raw instead
    'metacritic_url',
    'reddit_url',
    'reddit_name',
    'reddit_description',
    'reddit_logo',
    'website',
    'reactions',                    # opaque numeric ID map, not useful
]

df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
print(f"Remaining columns: {df.shape[1]}")

Remaining columns: 40


## 3. Normalize nulls
The dataset mixes `"nan"` strings, empty strings, and actual `NaN`.

In [6]:
# Replace string "nan" and empty strings with proper NaN
df.replace({'nan': np.nan, '': np.nan}, inplace=True)

# Verify
print("Null % after normalization:")
(df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False).head(15)

Null % after normalization:


reddit_count            99.8
additions_count         99.8
tba                     99.7
parents_count           99.6
metacritic_platforms    99.5
alternative_names       99.3
metacritic              99.2
twitch_count            98.9
reviews_text_count      98.9
game_series_count       98.8
youtube_count           98.7
movies_count            98.6
creators_count          98.3
rating                  98.0
rating_top              97.7
dtype: float64

## 3b. Filter to games with meaningful data
~98% of rows are bare-bones entries (shovelware, abandoned pages) with no rating, no genre, no release date. We keep only games that have enough signal to actually visualize.

In [7]:
n_before = len(df)

df = df[
    df['rating'].notna() &
    df['genres'].notna() &
    df['released'].notna() &
    (pd.to_numeric(df['ratings_count'], errors='coerce') > 10)  # skip games with <10 ratings
].copy()

n_after = len(df)
print(f"Filtered: {n_before:,} → {n_after:,} games ({n_after/n_before*100:.1f}% kept)")
print(f"Dropped {n_before - n_after:,} rows with insufficient data")

# Re-check null situation
print("\nNull % after filtering:")
(df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False).head(15)

Filtered: 889,793 → 12,207 games (1.4% kept)
Dropped 877,586 rows with insufficient data

Null % after filtering:


tba                          100.0
parents_count                 90.2
additions_count               89.9
reddit_count                  89.2
alternative_names             87.9
movies_count                  85.8
metacritic_platforms          75.4
game_series_count             64.4
creators_count                59.2
metacritic                    57.5
reviews_text_count            55.0
esrb_rating                   51.7
twitch_count                  42.3
parent_achievements_count     40.1
achievements_count            40.0
dtype: float64

## 4. Parse ESRB rating
Field looks like `{"id":4,"name":"Mature","slug":"mature"}` — extract just the name.

In [8]:
def parse_esrb(val):
    if pd.isna(val):
        return np.nan
    try:
        d = json.loads(val.replace('""', '"'))  # handle CSV double-quoting
        return d.get('name', np.nan)
    except:
        return np.nan

df['esrb_rating'] = df['esrb_rating'].apply(parse_esrb)
df['esrb_rating'].value_counts(dropna=False)

esrb_rating
NaN               6308
Teen              1990
Mature            1497
Everyone 10+      1239
Everyone           906
Adults Only        237
Rating Pending      30
Name: count, dtype: int64

## 5. Parse added_by_status → separate columns
Field looks like `{"beaten":1066,"dropped":178,"owned":9637,"playing":6,"toplay":80,"yet":709}`

In [9]:
def parse_json_safe(val):
    if pd.isna(val):
        return {}
    try:
        return json.loads(val.replace('""', '"'))
    except:
        return {}

status_df = df['added_by_status'].apply(parse_json_safe).apply(pd.Series)
status_cols = ['owned', 'beaten', 'playing', 'toplay', 'dropped', 'yet']
for col in status_cols:
    if col in status_df.columns:
        df[f'status_{col}'] = status_df[col]

df.drop(columns=['added_by_status'], inplace=True)
df[['name'] + [f'status_{c}' for c in status_cols if f'status_{c}' in df.columns]].head()

,name,status_owned,status_beaten,status_playing,status_toplay,status_dropped,status_yet
0,Half-Life 2: Lost Coast,9637.0,1066.0,6.0,80.0,178.0,709.0
1,Left 4 Dead 2,12885.0,2613.0,154.0,116.0,1236.0,394.0
2,Portal,11105.0,5216.0,86.0,285.0,422.0,468.0
3,Rocket League,9396.0,858.0,544.0,114.0,1681.0,194.0
4,Life is Strange,10297.0,3507.0,153.0,374.0,663.0,794.0


## 6. Parse ratings distribution
Field: `{count: 550, id: 4, percent: 47.13, title: recommended}|{count: 263, ...}`

Extract into: `rating_exceptional`, `rating_recommended`, `rating_meh`, `rating_skip` (as percentages).

In [10]:
def parse_ratings(val):
    result = {'exceptional': np.nan, 'recommended': np.nan, 'meh': np.nan, 'skip': np.nan}
    if pd.isna(val):
        return result
    for chunk in str(val).split('|'):
        # Extract title and percent with regex (the format isn't valid JSON)
        title_m = re.search(r'title:\s*(\w+)', chunk)
        pct_m = re.search(r'percent:\s*([\d.]+)', chunk)
        if title_m and pct_m:
            title = title_m.group(1)
            if title in result:
                result[title] = float(pct_m.group(1))
    return result

ratings_df = df['ratings'].apply(parse_ratings).apply(pd.Series)
for col in ratings_df.columns:
    df[f'pct_{col}'] = ratings_df[col]

df.drop(columns=['ratings'], inplace=True)
df[['name', 'pct_exceptional', 'pct_recommended', 'pct_meh', 'pct_skip']].head()

,name,pct_exceptional,pct_recommended,pct_meh,pct_skip
0,Half-Life 2: Lost Coast,14.57,47.13,22.54,15.77
1,Left 4 Dead 2,31.66,53.40,11.24,3.70
2,Portal,59.92,33.83,4.01,2.25
3,Rocket League,24.50,54.79,15.48,5.23
4,Life is Strange,43.92,36.58,13.20,6.30


## 7. Fix dtypes

In [11]:
# Release date → datetime
df['released'] = pd.to_datetime(df['released'], errors='coerce')
df['updated'] = pd.to_datetime(df['updated'], errors='coerce')

# Extract year for easy temporal analysis
df['release_year'] = df['released'].dt.year

# Numeric columns that might have been read as float due to NaN
int_candidates = [
    'id', 'ratings_count', 'reviews_count', 'reviews_text_count',
    'added', 'suggestions_count', 'reddit_count', 'youtube_count',
    'twitch_count', 'creators_count', 'screenshots_count',
    'achievements_count', 'parent_achievements_count',
    'game_series_count', 'additions_count', 'movies_count',
    'parents_count'
]
for col in int_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['metacritic'] = pd.to_numeric(df['metacritic'], errors='coerce')
df['playtime'] = pd.to_numeric(df['playtime'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['rating_top'] = pd.to_numeric(df['rating_top'], errors='coerce')
df['tba'] = pd.to_numeric(df['tba'], errors='coerce')

print(df.dtypes)

id                                    int64
slug                                    str
name                                    str
released                     datetime64[us]
rating                              float64
rating_top                          float64
ratings_count                       float64
reviews_text_count                  float64
added                               float64
playtime                            float64
suggestions_count                   float64
updated                      datetime64[us]
reviews_count                       float64
platforms                               str
stores                                  str
developers                              str
genres                                  str
tags                                    str
publishers                              str
esrb_rating                             str
name_original                           str
parents_count                       float64
reddit_count                    

## 8. Clean multi-value string fields
These stay as pipe-delimited strings in the main df (explode on demand for specific analyses).

In [12]:
MULTI_VALUE_COLS = ['platforms', 'parent_platforms', 'stores', 'developers',
                    'genres', 'tags', 'publishers']

# Strip whitespace from each value
for col in MULTI_VALUE_COLS:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: '|'.join(v.strip() for v in str(x).split('|')) if pd.notna(x) else np.nan
        )

# Count helpers — how many genres/platforms/tags per game
for col in ['genres', 'platforms', 'tags']:
    if col in df.columns:
        df[f'n_{col}'] = df[col].apply(lambda x: len(x.split('|')) if pd.notna(x) else 0)

df[['name', 'genres', 'n_genres', 'platforms', 'n_platforms', 'tags', 'n_tags']].head()

,name,genres,n_genres,platforms,n_platforms,tags,n_tags
0,Half-Life 2: Lost Coast,Action,1,macOS|Linux|PC,3,Singleplayer|Multiplayer|Atmospheric|Great Soundtrack|First-Person|Sci-fi|FP...,17
1,Left 4 Dead 2,Action|Shooter,2,Xbox 360|Linux|PC|macOS,4,Singleplayer|Steam Achievements|Multiplayer|Full controller support|Steam Cl...,28
2,Portal,Action|Puzzle,2,macOS|PC|Android|PlayStation 3|Xbox 360|Linux|Nintendo Switch,7,Singleplayer|Steam Achievements|Atmospheric|Great Soundtrack|Story Rich|Firs...,21
3,Rocket League,Sports|Racing|Indie,3,Nintendo Switch|Linux|macOS|Xbox One|PC|PlayStation 4,6,Singleplayer|Steam Achievements|Multiplayer|Full controller support|Steam Cl...,25
4,Life is Strange,Adventure,1,iOS|PC|Linux|PlayStation 3|macOS|Xbox 360|Android|PlayStation 4|Xbox One,9,Singleplayer|Full controller support|Atmospheric|steam-trading-cards|Great S...,24


## 9. Compute derived features

In [13]:
# Completion rate: beaten / (owned) where owned > 0
df['completion_rate'] = np.where(
    df['status_owned'] > 0,
    df['status_beaten'] / df['status_owned'],
    np.nan
)

# Drop rate: dropped / (owned) where owned > 0
df['drop_rate'] = np.where(
    df['status_owned'] > 0,
    df['status_dropped'] / df['status_owned'],
    np.nan
)

# Has metacritic score flag
df['has_metacritic'] = df['metacritic'].notna()

# Rating delta: RAWG community vs Metacritic (normalized to same 0-5 scale)
df['metacritic_norm'] = df['metacritic'] / 20  # 0-100 → 0-5
df['rating_delta'] = df['rating'] - df['metacritic_norm']

df[['name', 'completion_rate', 'drop_rate', 'rating', 'metacritic', 'rating_delta']].dropna().head()

,name,completion_rate,drop_rate,rating,metacritic,rating_delta
1,Left 4 Dead 2,0.202794,0.095925,4.09,89.0,-0.36
2,Portal,0.469698,0.038001,4.49,90.0,-0.01
3,Rocket League,0.091315,0.178906,3.93,86.0,-0.37
4,Life is Strange,0.340585,0.064388,4.12,83.0,-0.03
5,Counter-Strike: Global Offensive,0.079282,0.149123,3.57,81.0,-0.48


## 10. Drop remaining junk & reorder

In [14]:
# Drop columns that won't help with viz
FINAL_DROP = [
    'slug',                   # redundant with id+name
    'name_original',          # mostly same as name
    'rating_top',             # always 4 or 5, not informative
    'metacritic_platforms',   # sparse, complex
    'alternative_names',      # text, not useful for viz
    'description_raw',        # keep for NLP if needed, but heavy — drop for basic viz
    'tba',                    # almost entirely null
    'parents_count',          # sparse, unclear meaning
]

df.drop(columns=[c for c in FINAL_DROP if c in df.columns], inplace=True)
print(f"Final shape: {df.shape}")
df.columns.tolist()

Final shape: (12207, 49)


['id',
 'name',
 'released',
 'rating',
 'ratings_count',
 'reviews_text_count',
 'added',
 'playtime',
 'suggestions_count',
 'updated',
 'reviews_count',
 'platforms',
 'stores',
 'developers',
 'genres',
 'tags',
 'publishers',
 'esrb_rating',
 'reddit_count',
 'youtube_count',
 'creators_count',
 'twitch_count',
 'screenshots_count',
 'parent_platforms',
 'metacritic',
 'achievements_count',
 'parent_achievements_count',
 'game_series_count',
 'additions_count',
 'movies_count',
 'status_owned',
 'status_beaten',
 'status_playing',
 'status_toplay',
 'status_dropped',
 'status_yet',
 'pct_exceptional',
 'pct_recommended',
 'pct_meh',
 'pct_skip',
 'release_year',
 'n_genres',
 'n_platforms',
 'n_tags',
 'completion_rate',
 'drop_rate',
 'has_metacritic',
 'metacritic_norm',
 'rating_delta']

## 11. Quick sanity checks

In [15]:
print("=== Duplicates ===")
print(f"  By id: {df['id'].duplicated().sum()}")
print(f"  By name: {df['name'].duplicated().sum()}")

print("\n=== Key distributions ===")
print(f"  Release years: {df['release_year'].min():.0f} – {df['release_year'].max():.0f}")
print(f"  Metacritic: {df['metacritic'].describe()[['count','mean','min','max']].to_dict()}")
print(f"  Rating: {df['rating'].describe()[['count','mean','min','max']].to_dict()}")
print(f"  Playtime: median={df['playtime'].median()}, max={df['playtime'].max()}")

print("\n=== ESRB breakdown ===")
print(df['esrb_rating'].value_counts(dropna=False).head(10))

print("\n=== Top genres ===")
genre_counts = df['genres'].dropna().str.split('|').explode().str.strip().value_counts()
print(genre_counts.head(15))

print("\n=== Null % in final df ===")
(df.isna().sum() / len(df) * 100).round(1).sort_values(ascending=False).head(15)

=== Duplicates ===
  By id: 0
  By name: 1

=== Key distributions ===
  Release years: 1962 – 2026
  Metacritic: {'count': 5189.0, 'mean': 74.39140489497012, 'min': 21.0, 'max': 99.0}
  Rating: {'count': 12207.0, 'mean': 3.427257311378717, 'min': 1.0, 'max': 4.82}
  Playtime: median=3.0, max=900.0

=== ESRB breakdown ===
esrb_rating
NaN               6308
Teen              1990
Mature            1497
Everyone 10+      1239
Everyone           906
Adults Only        237
Rating Pending      30
Name: count, dtype: int64

=== Top genres ===
genres
Action                   6286
Adventure                4388
Indie                    4304
RPG                      2629
Strategy                 2212
Casual                   1810
Simulation               1605
Shooter                  1258
Arcade                   1050
Puzzle                    826
Platformer                710
Racing                    656
Sports                    652
Massively Multiplayer     387
Fighting                  374
N

additions_count              89.9
reddit_count                 89.2
movies_count                 85.8
game_series_count            64.4
creators_count               59.2
metacritic                   57.5
metacritic_norm              57.5
rating_delta                 57.5
reviews_text_count           55.0
esrb_rating                  51.7
twitch_count                 42.3
parent_achievements_count    40.1
achievements_count           40.0
youtube_count                37.4
status_playing               26.6
dtype: float64

## 12. Export

In [16]:
# Parquet is smaller and preserves dtypes
df.to_parquet('rawg_clean.parquet', index=False)

# Also CSV if needed
# df.to_csv('rawg_clean.csv', index=False)

print(f"Exported {len(df)} rows × {df.shape[1]} cols to rawg_clean.parquet")

Exported 12207 rows × 49 cols to rawg_clean.parquet
